In [1]:
import requests
import os, math, time
from dotenv import load_dotenv
import pandas as pd

In [2]:
load_dotenv()
API_KEY = os.getenv('TAGO_API_KEY')

if API_KEY:
    print(f'API 키 로드 완료: {API_KEY[:6]}...{API_KEY[-4:]}')
else:
    print('env 파일에 API 키를 설정하세요')

API 키 로드 완료: 0f4cc2...0ab9


In [3]:
url = 'https://apis.data.go.kr/B553077/api/open/sdsc2/storeListInUpjong'
params = {
    'ServiceKey':API_KEY,
    'pageNo':1,
    'numOfRows':10,
    'divId':'indsSclsCd',
    'key':'G20405',
    'type':'json'
}
response = requests.get(url, params)
response.status_code

200

In [4]:
data = response.json()
data

{'header': {'description': '소상공인시장진흥공단 주요상권내 상가업소정보',
  'columns': ['상가업소번호',
   '상호명',
   '지점명',
   '상권업종대분류코드',
   '상권업종대분류명',
   '상권업종중분류코드',
   '상권업종중분류명',
   '상권업종소분류코드',
   '상권업종소분류명',
   '표준산업분류코드',
   '표준산업분류명',
   '시도코드',
   '시도명',
   '시군구코드',
   '시군구명',
   '행정동코드',
   '행정동명',
   '법정동코드',
   '법정동명',
   'PNU코드',
   '대지구분코드',
   '대지구분명',
   '지번본번지',
   '지번부번지',
   '지번주소',
   '도로명코드',
   '도로명',
   '건물본번지',
   '건물부번지',
   '건물관리번호',
   '건물명',
   '도로명주소',
   '구우편번호',
   '신우편번호',
   '동정보',
   '층정보',
   '호정보',
   '경도',
   '위도'],
  'stdrYm': '202603',
  'resultCode': '00',
  'resultMsg': 'NORMAL SERVICE'},
 'body': {'items': [{'bizesId': 'MA010120220800006053',
    'bizesNm': 'GS25한남',
    'brchNm': '제일점',
    'indsLclsCd': 'G2',
    'indsLclsNm': '소매',
    'indsMclsCd': 'G204',
    'indsMclsNm': '종합 소매',
    'indsSclsCd': 'G20405',
    'indsSclsNm': '편의점',
    'ksicCd': 'G47122',
    'ksicNm': '체인화 편의점',
    'ctprvnCd': '11',
    'ctprvnNm': '서울특별시',
    'signguCd': '11170',
    'sign

In [6]:
data_body = data['body']
data_body

{'items': [{'bizesId': 'MA010120220800006053',
   'bizesNm': 'GS25한남',
   'brchNm': '제일점',
   'indsLclsCd': 'G2',
   'indsLclsNm': '소매',
   'indsMclsCd': 'G204',
   'indsMclsNm': '종합 소매',
   'indsSclsCd': 'G20405',
   'indsSclsNm': '편의점',
   'ksicCd': 'G47122',
   'ksicNm': '체인화 편의점',
   'ctprvnCd': '11',
   'ctprvnNm': '서울특별시',
   'signguCd': '11170',
   'signguNm': '용산구',
   'adongCd': '11170685',
   'adongNm': '한남동',
   'ldongCd': '1117013100',
   'ldongNm': '한남동',
   'lnoCd': '1117013100106390001',
   'plotSctCd': '1',
   'plotSctNm': '대지',
   'lnoMnno': 639,
   'lnoSlno': 1,
   'lnoAdr': '서울특별시 용산구 한남동 639-1',
   'rdnmCd': '111703102001',
   'rdnm': '서울특별시 용산구 대사관로',
   'bldMnno': 66,
   'bldSlno': '',
   'bldMngNo': '1117013100106390001006288',
   'bldNm': '',
   'rdnmAdr': '서울특별시 용산구 대사관로 66',
   'oldZipcd': '140887',
   'newZipcd': '04402',
   'dongNo': '',
   'flrNo': '',
   'hoNo': '',
   'lon': 127.005340193519,
   'lat': 37.5331737188111},
  {'bizesId': 'MA01012022080000187

In [7]:
total_count = data_body['totalCount']
total_pages = math.ceil(total_count / 1000)
total_pages

56

In [8]:
all_items = []

for page_no in range(1, total_pages+1):
    params={
        'ServiceKey':API_KEY,
        'pageNo':page_no,
        'numOfRows':1000,
        'divId':'indsSclsCd',
        'key':'G20405',
        'type':'json'
    }

    response = requests.get(url=url, params=params, timeout=30)
    response.raise_for_status()

    page_body = response.json()['body']

    if not page_body.get('items'):
        print(f'{page_no} / {total_pages} 페이지: 페이지 없음')
        continue

    page_items = page_body.get('items', []) # 없으면 빈 리스트

    if isinstance(page_items, dict):
        page_items = [page_items]   # 딕셔너리라면 리스트로 저장
    
    all_items.extend(page_items)    # 여러 개를 맨 뒤에 한꺼번에 추가하는 리스트 함수

    print(f'{page_no} / {total_pages} 페이지 수집 완료 → 누적 {len(all_items):,}건')

    time.sleep(0.2) # 너무 빠르게 요청하지 않게 0.2초 기다린 후 다음 진행

print(f'전체 수집 완료 : {len(all_items):,}건')

1 / 56 페이지 수집 완료 → 누적 1,000건
2 / 56 페이지 수집 완료 → 누적 2,000건
3 / 56 페이지 수집 완료 → 누적 3,000건
4 / 56 페이지 수집 완료 → 누적 4,000건
5 / 56 페이지 수집 완료 → 누적 5,000건
6 / 56 페이지 수집 완료 → 누적 6,000건
7 / 56 페이지 수집 완료 → 누적 7,000건
8 / 56 페이지 수집 완료 → 누적 8,000건
9 / 56 페이지 수집 완료 → 누적 9,000건
10 / 56 페이지 수집 완료 → 누적 10,000건
11 / 56 페이지 수집 완료 → 누적 11,000건
12 / 56 페이지 수집 완료 → 누적 12,000건
13 / 56 페이지 수집 완료 → 누적 13,000건
14 / 56 페이지 수집 완료 → 누적 14,000건
15 / 56 페이지 수집 완료 → 누적 15,000건
16 / 56 페이지 수집 완료 → 누적 16,000건
17 / 56 페이지 수집 완료 → 누적 17,000건
18 / 56 페이지 수집 완료 → 누적 18,000건
19 / 56 페이지 수집 완료 → 누적 19,000건
20 / 56 페이지 수집 완료 → 누적 20,000건
21 / 56 페이지 수집 완료 → 누적 21,000건
22 / 56 페이지 수집 완료 → 누적 22,000건
23 / 56 페이지 수집 완료 → 누적 23,000건
24 / 56 페이지 수집 완료 → 누적 24,000건
25 / 56 페이지 수집 완료 → 누적 25,000건
26 / 56 페이지 수집 완료 → 누적 26,000건
27 / 56 페이지 수집 완료 → 누적 27,000건
28 / 56 페이지 수집 완료 → 누적 28,000건
29 / 56 페이지 수집 완료 → 누적 29,000건
30 / 56 페이지 수집 완료 → 누적 30,000건
31 / 56 페이지 수집 완료 → 누적 31,000건
32 / 56 페이지 수집 완료 → 누적 32,000건
33 / 56 페이지 수집 완료 → 누적 33,

In [9]:
df = pd.DataFrame(all_items)
df.head()

,bizesId,bizesNm,brchNm,indsLclsCd,indsLclsNm,indsMclsCd,indsMclsNm,indsSclsCd,indsSclsNm,ksicCd,...,bldMngNo,bldNm,rdnmAdr,oldZipcd,newZipcd,dongNo,flrNo,hoNo,lon,lat
0,MA010120220800137300,씨유중구정동길점,,G2,소매,G204,종합 소매,G20405,편의점,G47122,...,1114016700100270012000002,,서울특별시 중구 정동길 10,100070,04516,,,,126.969687,37.567480
1,MA010120220800068742,세븐혜화점주,코리아,G2,소매,G204,종합 소매,G20405,편의점,G47122,...,1111016900101110007007679,,서울특별시 종로구 창경궁로 273,110530,03075,,,,127.000600,37.585792
2,MA010120220800524824,CU군산조촌,,G2,소매,G204,종합 소매,G20405,편의점,G47122,...,4513013400108480000027153,,전북특별자치도 군산시 조촌안2길 25,573885,54076,,,,126.734879,35.971339
3,MA010120220800443237,씨유중대동양점,,G2,소매,G204,종합 소매,G20405,편의점,G47122,...,1159010500101900035017575,,서울특별시 동작구 흑석로9길 7,156861,06910,,1,,126.959736,37.507597
4,MA010120220800449645,GS25대방역점,,G2,소매,G204,종합 소매,G20405,편의점,G47122,...,1156013200114320000006001,봉덕빌딩,서울특별시 영등포구 여의대방로62길 2,150855,07319,,1,,126.926670,37.513816


In [11]:
df.to_excel('convinience_list.xlsx', index=False)